## Step 1: Install the comment downloader

**What happens:** `youtube-comment-downloader` reads public YouTube comments directly, so no API key, Google Cloud account, or billing is needed.

In [1]:
!pip install youtube-comment-downloader

## Step 2: Create `youtube_ingest.py`

**What happens:**
- `extract_video_id()` pulls the 11-character ID from `watch?v=`, `youtu.be/`, or `shorts/` links.
- `fetch_comments()` streams comments and stops at `max_comments` using `islice`.
- `%%writefile` saves the cell as a module that `app.py` can import.

In [2]:
%%writefile youtube_ingest.py
import re
from itertools import islice
from youtube_comment_downloader import YoutubeCommentDownloader

def extract_video_id(url):
    m = re.search(r"(?:v=|youtu\.be/|shorts/)([A-Za-z0-9_-]{11})", url)
    return m.group(1) if m else None

def fetch_comments(url, max_comments=100):
    vid = extract_video_id(url)
    if not vid:
        raise ValueError("Invalid YouTube URL")
    dl = YoutubeCommentDownloader()
    gen = dl.get_comments_from_url(f"https://www.youtube.com/watch?v={vid}")
    return vid, [c["text"] for c in islice(gen, max_comments)]

Writing youtube_ingest.py


## Fix: SSL certificate error

**What happens:** `pip-system-certs` makes `requests` use the Windows certificate store, which already trusts your network's certificate. After installing, **restart the kernel** (and the Flask server) so it takes effect.

In [5]:
!pip install pip-system-certs certifi --upgrade

  Using cached pip_system_certs-5.3-py3-none-any.whl.metadata (3.9 kB)
Using cached pip_system_certs-5.3-py3-none-any.whl (6.9 kB)


## Retry the comment fetch

**What happens:** Same test as before after the kernel restart. Expected: the video ID, `5`, and five comment previews.

In [7]:
from youtube_ingest import fetch_comments

vid, comments = fetch_comments("https://youtu.be/FW_PPcr1sPE", 5)
print(vid, len(comments))
for c in comments:
    print("-", c[:80])

SSLError: HTTPSConnectionPool(host='www.youtube.com', port=443): Max retries exceeded with url: /watch?v=FW_PPcr1sPE (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1082)')))

## Fallback: disable SSL verification (dev only)

**What happens:** We set `verify = False` on the downloader's session so the blocked certificate check is skipped. Acceptable for local testing of public comments only.

In [8]:
import urllib3
urllib3.disable_warnings()

from itertools import islice
from youtube_comment_downloader import YoutubeCommentDownloader

dl = YoutubeCommentDownloader()
dl.session.verify = False
gen = dl.get_comments_from_url("https://www.youtube.com/watch?v=FW_PPcr1sPE")
for c in islice(gen, 5):
    print("-", c["text"][:80])

- Mohanlal ❤
- Lalettan mass aanu 🔥
- 💥🔥
- Lalettan 🔥
- 🎉🎉🎉🎉🎉🎉🎉❤❤❤❤❤தலைவா தலைவா சூப்பர்


## Update `youtube_ingest.py`

**What happens:**
- Sets `verify = False` on the downloader session (dev-only workaround for the network SSL block).
- Skips comments with no letters (emoji-only), since they aren't useful for text sentiment.
- Fetches extra comments first, so we still return up to `max_comments` after filtering.

In [10]:
%%writefile youtube_ingest.py
import re, urllib3
from itertools import islice
from youtube_comment_downloader import YoutubeCommentDownloader

urllib3.disable_warnings()

def extract_video_id(url):
    m = re.search(r"(?:v=|youtu\.be/|shorts/)([A-Za-z0-9_-]{11})", url)
    return m.group(1) if m else None

def has_text(s):
    return any(ch.isalpha() for ch in s)

def fetch_comments(url, max_comments=100):
    vid = extract_video_id(url)
    if not vid:
        raise ValueError("Invalid YouTube URL")
    dl = YoutubeCommentDownloader()
    dl.session.verify = False   # dev only
    gen = dl.get_comments_from_url(f"https://www.youtube.com/watch?v={vid}")
    out = []
    for c in islice(gen, max_comments * 3):
        if has_text(c["text"]):
            out.append(c["text"])
        if len(out) >= max_comments:
            break
    return vid, out

Overwriting youtube_ingest.py


## Test the `/youtube` endpoint

**What happens:** Restart Flask so it reloads the updated module, then POST the video URL. Expected: status `200` and a summary like `{'Positive': 12, 'Neutral': 5, 'Negative': 3}`.

In [11]:
import requests

r = requests.post("http://127.0.0.1:5000/youtube",
    json={"url": "https://youtu.be/FW_PPcr1sPE", "max_comments": 20})
print(r.status_code, r.json()["summary"])

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /youtube (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=5000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))

## Start the Flask server

**What happens:** Flask must be running in its own terminal while the notebook sends requests to it. Make sure `app.py` already contains the `/youtube` route (Step 4 earlier) and the imports `from collections import Counter` and `from youtube_ingest import fetch_comments`. Wait for the line `Running on http://127.0.0.1:5000` before testing.

## Health check

**What happens:** Confirms the notebook can reach the Flask server. Expected: `200`.

In [12]:
import requests
print(requests.get("http://127.0.0.1:5000/health").status_code)

200


## Test the `/youtube` endpoint

**What happens:** Sends the video URL, fetches 20 comments, classifies each one, and prints the sentiment summary plus a few labelled comments. Expected: status `200` and a summary like `{'Positive': 12, 'Neutral': 5, 'Negative': 3}`.

In [13]:
r = requests.post("http://127.0.0.1:5000/youtube",
    json={"url": "https://youtu.be/FW_PPcr1sPE", "max_comments": 20},
    timeout=300)
d = r.json()
print(r.status_code, d.get("summary") or d)
for x in d.get("results", [])[:5]:
    print(x["label"], "|", x["text"][:60])

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

## Debug: see the raw response

**What happens:** Prints the status code and first 300 characters of the reply instead of parsing JSON. `404` means the route is missing from `app.py`; `500` means a server-side error (the Flask terminal will show it).

In [14]:
print(r.status_code)
print(r.text[:300])

404
<!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.</p>



## Add the `/youtube` route to `app.py` automatically

**What happens:**
- Reads `app.py` and checks the route isn't already there.
- Inserts the `/youtube` route just above `if __name__ == "__main__":`.
- Adds the two imports it needs at the top.

After running it, **restart Flask** (Ctrl+C in its terminal, then `python app.py`).

In [15]:
src = open("app.py", encoding="utf-8").read()

if '"/youtube"' in src:
    print("Route already exists")
else:
    route = '''
@app.route("/youtube", methods=["POST"])
def youtube():
    data = request.get_json(silent=True) or {}
    url = data.get("url", "")
    n = min(int(data.get("max_comments", 100)), 500)
    try:
        vid, comments = fetch_comments(url, n)
    except Exception as e:
        return jsonify({"error": str(e)}), 400
    results = []
    for c in comments:
        res = predict_sentiment(c)
        results.append({"text": c, "label": res["label"], "scores": res["scores"]})
    summary = dict(Counter(x["label"] for x in results))
    return jsonify({"video_id": vid, "total": len(results),
                    "summary": summary, "results": results})

'''
    marker = 'if __name__ == "__main__":'
    assert marker in src, "main block not found"
    src = src.replace(marker, route + marker, 1)
    src = "from collections import Counter\nfrom youtube_ingest import fetch_comments\n" + src
    open("app.py", "w", encoding="utf-8").write(src)
    print("Route added")

Route added


## Re-test `/youtube` (after restarting Flask)

**What happens:** Same request as before. Expected: status `200`, a sentiment summary, and 5 labelled comments.

In [16]:
import requests
r = requests.post("http://127.0.0.1:5000/youtube",
    json={"url": "https://youtu.be/FW_PPcr1sPE", "max_comments": 20},
    timeout=300)
print(r.status_code)
d = r.json()
print(d.get("summary") or d)
for x in d.get("results", [])[:5]:
    print(x["label"], "|", x["text"][:60])

404


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

## Check that the route is in `app.py`

**What happens:** Confirms the file on disk contains the `/youtube` route. If it prints `False`, rerun the insert cell from the previous step.

In [17]:
src = open("app.py", encoding="utf-8").read()
print('"/youtube"' in src)

True


## Restart Flask, then see the raw reply

**What happens:** Flask only loads `app.py` at startup, so first press **Ctrl+C** in the Flask terminal and run `python app.py` again. Wait for `Running on http://127.0.0.1:5000`, then run this cell. It prints the status and raw text instead of parsing JSON.

In [18]:
import requests
r = requests.post("http://127.0.0.1:5000/youtube",
    json={"url": "https://youtu.be/FW_PPcr1sPE", "max_comments": 20},
    timeout=300)
print(r.status_code)
print(r.text[:300])

404
<!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.</p>



## Inspect `app.py`

**What happens:** Prints every route in the file and the line numbers of the `/youtube` route and `app.run`. The route must appear **before** `app.run(...)`, otherwise Flask never registers it.

In [19]:
for i, line in enumerate(open("app.py", encoding="utf-8").read().splitlines(), 1):
    if "@app.route" in line or "app.run" in line or "__main__" in line:
        print(i, line)

39 @app.route("/youtube", methods=["POST"])
56 if __name__ == "__main__":
57     app.run(host="127.0.0.1", port=5000, debug=False)


## Find and stop the old server on port 5000

**What happens:** An old Flask process may still be holding port 5000, so your restart silently failed to bind and you keep hitting the old version. This lists what is using the port and kills it. Then start `python app.py` again.

## Restart Flask

**What happens:** Starts the updated `app.py` in a fresh process. Wait for `Running on http://127.0.0.1:5000`, then keep the window open. If the model loads but no route list appears, that's normal.

## Step 22: Test the `/youtube` route

**What happens:** Sends a YouTube link to your running Flask app and prints the summary. Start with a small `max_comments` so it's quick and uses little API quota.

In [3]:
import requests, json
print(requests.get("http://127.0.0.1:5000/health").json())

{'status': 'ok'}


In [5]:
r = requests.post(URL, json=payload, timeout=120)
print("Status:", r.status_code)
print(r.text[:1500])

Status: 200
{"results":[],"summary":{},"total":0,"video_id":"yLKKz4-M_y8"}



In [4]:
import requests, json

URL = "http://127.0.0.1:5000/youtube"
payload = {
    "url": "https://youtu.be/yLKKz4-M_y8?list=RDyLKKz4-M_y8",
    "max_comments": 20
}

r = requests.post(URL, json=payload, timeout=120)
print("Status:", r.status_code)
print(json.dumps(r.json(), indent=2, ensure_ascii=False)[:2000])

Status: 500


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

## Step 22b: Test the comment scraper directly

**What happens:** Bypasses Flask and calls `youtube-comment-downloader` itself, so we can see whether comments are being fetched at all.

In [6]:
from youtube_comment_downloader import YoutubeCommentDownloader
from itertools import islice

d = YoutubeCommentDownloader()
comments = list(islice(d.get_comments_from_url("https://www.youtube.com/watch?v=yLKKz4-M_y8"), 5))
print(len(comments))
for c in comments:
    print(c["text"][:100])

5
Ranabaali looks like such an exciting combination of Vijay Deverakonda and Rashmika, The Telugu surp
AJAY ATUL Music ❤
Ajay-atul as a maharashtrian,  you nailed it bro❤️‍🔥❤️‍🔥❤️‍🔥
This is song super in india ❤❤❤ super and super love is song
❤❤❤❤


In [7]:
r = requests.post(URL, json=payload, timeout=300)
print("Status:", r.status_code)
print(r.text[:1500])

Status: 500
<!doctype html>
<html lang=en>
<title>500 Internal Server Error</title>
<h1>Internal Server Error</h1>
<p>The server encountered an internal error and was unable to complete your request. Either the server is overloaded or there is an error in the application.</p>

